# Benchmark: our operator with an SFNO-style encoder/decoder

`sphere_ours_vs_sfno.ipynb` showed that our model, given the exact spherical
harmonic basis, still trailed SFNO by 10-25x. The model was undertrained rather
than limited by the basis: it is narrow (8 latent channels) and its spectral
weights are hard to optimise. SFNO wraps its spectral layers in a pointwise
encoder and decoder. This notebook adds the same structure around our
`IntegralOperator` layers (`EncoderDecoderOperatorModel` in `model.py`) and
reruns the manifold benchmark with ODEs and PDEs:

| model | structure |
|---|---|
| **Ours** | `IntegralOperatorModel` exactly as in `benchmark_manifolds_v2` (4 layers, 8 latent channels, LeakyReLU) |
| **Ours-ED** | Linear encoder `in -> width`, 4 x `IntegralOperator(width -> width)` with GELU, decoder MLP `width -> 128 -> out` (SFNO's `fc0` / `fc1`, `fc2`) |
| **Ours-ED+skip** | Ours-ED plus a pointwise Linear skip in every block (SFNO's `w0..w3`) |
| CNO, FNO, LNO | 1D baselines on the vertex ordering, as in v2 |
| SFNO | Sphere only, as in v2 |
| FNO2d | Torus only, as in v2 |

Domains, meshes, Laplace-Beltrami bases, initial conditions, normalisation and
the training protocol are unchanged from `benchmark_manifolds_v2.ipynb`.

**Systems** (input $u(0)$, target $u(T)$):

| system | equation |
|---|---|
| Pendulum (ODE) | $\ddot x + c\dot x + \sin x = 0$ at every vertex, $\dot x(0)=0$ |
| Duffing (ODE) | $\ddot x + c\dot x + x + x^3 = 0$ at every vertex, $\dot x(0)=0$ |
| Diffusion | $u_t = D\Delta u$ |
| Reaction-Diffusion | $u_t = D\Delta u + k u^2$ |
| Beam | $w_{tt} = -EI\,\Delta^2 w / \rho A$, $w_t(0)=0$ |

**Configuration by environment variable:**

| variable | default | meaning |
|---|---|---|
| `IC_TYPE` | `noise` | `noise` or `lowmodes`, as in v2 |
| `MODELS` | all | comma list from `Ours,Ours-ED,Ours-ED+skip,CNO,FNO,LNO,SFNO,FNO2d` |
| `DOMAINS` | `Sphere,Torus` | which domains to run |
| `SYSTEMS` | all | comma list from `Pendulum,Duffing,Diffusion,Reaction-Diffusion,Beam` |
| `N_EPOCHS` | 1000 | |
| `OURS_ED_WIDTH` | 32 | latent width of Ours-ED (SFNO uses 64; our cost per layer grows with width x V^2) |
| `OURS_MICRO_BATCH` | 16 | gradient-accumulation chunk for Ours |
| `OURS_ED_MICRO_BATCH` | 8 | same for Ours-ED variants (about 10 GiB at width 32) |
| `BENCHMARK_SMOKE` | unset | `1` = tiny data, 2 epochs |

Every job saves `results/ed_benchmark_<IC_TYPE>_<job id>.json`. The last section
merges all such files, so the run can be split across several jobs.

In [ ]:
import os
import glob
import json
import time
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import trimesh
from scipy.sparse.linalg import eigsh, expm_multiply

from equations import (
    Equation,
    MeshDomain,
    DiffusionEquation,
    ReactionDiffusion,
    BeamEquation,
    DrivenPendulum,
    DuffingOscillator,
)
from model import HeatKernel, IntegralOperator, IntegralOperatorModel, EncoderDecoderOperatorModel
from ref_models.cno import CNOOperator
from ref_models.fno import FNOOperator
from ref_models.lno import LNOOperator
from ref_models.sfno import SphereSFNOOperator
from ref_models.fno2d import TorusFNO2dOperator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Configuration

In [ ]:
def env_list(name, allowed):
    chosen = [x.strip() for x in os.environ.get(name, ",".join(allowed)).split(",") if x.strip()]
    unknown = set(chosen) - set(allowed)
    if unknown:
        raise ValueError(f"{name}: unknown {sorted(unknown)}; choose from {allowed}")
    return [x for x in allowed if x in chosen]


SEED = 0
N_TOTAL = 128
N_TRAIN = 96
N_EPOCHS = int(os.environ.get("N_EPOCHS", "1000"))
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

IC_TYPE = os.environ.get("IC_TYPE", "noise")
if IC_TYPE not in ("noise", "lowmodes"):
    raise ValueError(f"IC_TYPE must be 'noise' or 'lowmodes', got {IC_TYPE!r}")

ALL_MODELS = ["Ours", "Ours-ED", "Ours-ED+skip", "CNO", "FNO", "LNO", "SFNO", "FNO2d"]
ALL_DOMAINS = ["Sphere", "Torus"]
ALL_SYSTEMS = ["Pendulum", "Duffing", "Diffusion", "Reaction-Diffusion", "Beam"]
MODELS = env_list("MODELS", ALL_MODELS)
DOMAIN_NAMES = env_list("DOMAINS", ALL_DOMAINS)
SYSTEM_NAMES = env_list("SYSTEMS", ALL_SYSTEMS)

# Meshes - identical to benchmark_manifolds_v2.ipynb
SPHERE_SUBDIVISIONS = 3             # icosphere: 642 vertices
TORUS_MAJOR_RADIUS = 1.0
TORUS_MINOR_RADIUS = 0.3
TORUS_MAJOR_SECTIONS = 40           # 40 x 16 = 640 vertices
TORUS_MINOR_SECTIONS = 16
VERTEX_ORDER = "raster"

# Initial conditions - identical to v2
N_IC_MODES = 15
NOISE_SMOOTHING_TIME = 0.02
IC_BASIS_MODES = 96

# Our models
N_MODES = 96
N_DIFFUSION_TIMES = 12
HIDDEN_CHANNELS = 32                # phi MLP hidden size (all Ours variants)
LATENT_CHANNELS = 8                 # Ours
N_OPERATOR_LAYERS = 4
OURS_ED_WIDTH = int(os.environ.get("OURS_ED_WIDTH", "32"))
OURS_ED_DECODER_HIDDEN = 128        # as SFNO fc1
OURS_MICRO_BATCH = int(os.environ.get("OURS_MICRO_BATCH", "16"))
OURS_ED_MICRO_BATCH = int(os.environ.get("OURS_ED_MICRO_BATCH", "8"))

# Geometry-aware baselines - identical to v2
SFNO_NLAT, SFNO_MODES, SFNO_WIDTH = 32, 16, 64
FNO2D_MODES, FNO2D_WIDTH = 12, 32

ADD_COORDINATE_CHANNELS = True

# Physical parameters: PDEs identical to v2; ODEs as in sphere_ours_vs_sfno.ipynb
PDE_SETTINGS = dict(D=0.1, k=0.5, T=1.0, EI=0.04, rho_A=1.0, T_beam=1.0)
ODE_SETTINGS = {
    "Pendulum": dict(c=0.2, amplitude=2.0, T=3.0),
    "Duffing":  dict(c=0.2, amplitude=1.0, T=3.0),
}

if os.environ.get("BENCHMARK_SMOKE") == "1":
    N_TOTAL, N_TRAIN, N_EPOCHS, BATCH_SIZE = 12, 8, 2, 4
    print("SMOKE TEST MODE")

run_id = os.environ.get("SLURM_JOB_ID", "local")
print(f"IC_TYPE = {IC_TYPE}  N_EPOCHS = {N_EPOCHS}  run_id = {run_id}")
print(f"MODELS  = {MODELS}\nDOMAINS = {DOMAIN_NAMES}\nSYSTEMS = {SYSTEM_NAMES}")
print(f"OURS_ED_WIDTH = {OURS_ED_WIDTH}  micro-batches: Ours {OURS_MICRO_BATCH}, Ours-ED {OURS_ED_MICRO_BATCH}")

## Domains (unchanged from v2)

In [ ]:
def reorder_mesh(vertices, faces, order):
    inverse = np.empty_like(order)
    inverse[order] = np.arange(len(order))
    return vertices[order], inverse[faces]


def snake_order(band, angle):
    key = np.where(band % 2 == 0, angle, -angle)
    return np.lexsort((key, band))


def sphere_raster(vertices):
    polar = np.arccos(np.clip(vertices[:, 2] / np.linalg.norm(vertices, axis=1), -1, 1))
    azimuth = np.arctan2(vertices[:, 1], vertices[:, 0])
    n_bands = int(round(np.sqrt(len(vertices))))
    band = np.minimum((polar / np.pi * n_bands).astype(int), n_bands - 1)
    return snake_order(band, azimuth)


def torus_raster(vertices, n_major):
    major = np.mod(np.arctan2(vertices[:, 1], vertices[:, 0]), 2 * np.pi)
    rho = np.hypot(vertices[:, 0], vertices[:, 1])
    minor = np.arctan2(vertices[:, 2], rho - TORUS_MAJOR_RADIUS)
    band = np.round(major / (2 * np.pi) * n_major).astype(int) % n_major
    return snake_order(band, minor)


def build_domain(name):
    if name == "Sphere":
        mesh = trimesh.creation.icosphere(subdivisions=SPHERE_SUBDIVISIONS, radius=1.0)
        order_fn = sphere_raster
    else:
        mesh = trimesh.creation.torus(
            major_radius=TORUS_MAJOR_RADIUS,
            minor_radius=TORUS_MINOR_RADIUS,
            major_sections=TORUS_MAJOR_SECTIONS,
            minor_sections=TORUS_MINOR_SECTIONS,
        )
        order_fn = lambda v: torus_raster(v, TORUS_MAJOR_SECTIONS)

    vertices = np.asarray(mesh.vertices, dtype=float)
    faces = np.asarray(mesh.faces)
    if VERTEX_ORDER == "raster":
        vertices, faces = reorder_mesh(vertices, faces, order_fn(vertices))
    return MeshDomain(vertices, faces)


domains = {name: build_domain(name) for name in DOMAIN_NAMES}
for name, dom in domains.items():
    print(f"{name:7s} vertices {dom.n_points:4d}  faces {len(dom.faces):5d}  area {dom.mass_diagonal.sum():.3f}")

## Laplace-Beltrami eigenpairs (unchanged from v2)

In [ ]:
def make_mesh_eigenbasis(domain, n_modes):
    mass = domain.mass_diagonal.astype(np.float64)
    M_inv_sqrt = sp.diags(1.0 / np.sqrt(mass))
    L_symmetric = (M_inv_sqrt @ domain.stiffness @ M_inv_sqrt).tocsc()
    # fixed v0 makes the basis inside degenerate eigenspaces reproducible
    values, vectors = eigsh(L_symmetric, k=n_modes, sigma=-1e-6, which="LM",
                            v0=np.ones(L_symmetric.shape[0]))
    order = np.argsort(values)
    values = np.maximum(values[order], 0.0)
    vectors = np.asarray(M_inv_sqrt @ vectors[:, order])
    return values, vectors, mass


def make_diffusion_times(eigenvalues, n=N_DIFFUSION_TIMES):
    first_nonzero = eigenvalues[eigenvalues > 1e-8][0]
    return list(np.logspace(np.log10(1.0 / eigenvalues[-1]), np.log10(3.0 / first_nonzero), n))


bases = {name: make_mesh_eigenbasis(dom, N_MODES) for name, dom in domains.items()}
for name, (values, vectors, mass) in bases.items():
    error = np.linalg.norm(vectors.T @ (mass[:, None] * vectors) - np.eye(vectors.shape[1]))
    print(f"{name}: {vectors.shape[1]} modes  first eigenvalues {np.round(values[:6], 3)}  "
          f"lambda_max {values[-1]:.1f}  M-orthogonality error {error:.1e}")

## Data generation

PDE data use the same code and random stream as v2. The ODEs run independently
at every vertex, starting at rest from `amplitude * random_surface_field`.

The diagnostics are the same as in v2. "outside 96 modes" is the part of $u(T)$
outside our basis. Plain `Ours` cannot go below it, because its output lies in
that span. `Ours-ED` is only partly limited: every block's output is still
band-limited, so detail of $u(0)$ above the 96 modes is lost in the first
block, and the pointwise decoder can only add nonlinear functions of
band-limited fields. `Ours-ED+skip` carries the full-resolution input through
its pointwise skips, the way SFNO and FNO do. With `IC_TYPE=noise` this floor is
large for the ODEs and Beam (about 20-30%), so compare those rows with that in
mind, or run `IC_TYPE=lowmodes` too.

In [ ]:
class PointwiseODE(Equation):
    # An ODE from equations.py run independently at every vertex.
    # state = [x at all vertices, xdot at all vertices]
    def __init__(self, ode):
        super().__init__()
        self.ode = ode

    def evolution(self, t, state):
        return self.ode.evolution(t, state.reshape(2, -1)).ravel()


def random_surface_field(rng, domain_name):
    domain = domains[domain_name]
    _, eigenvectors, mass = bases[domain_name]
    if IC_TYPE == "lowmodes":
        field = eigenvectors[:, 1:N_IC_MODES + 1] @ rng.normal(size=N_IC_MODES)
    else:
        noise = rng.normal(size=domain.n_points)
        field = expm_multiply(NOISE_SMOOTHING_TIME * domain.laplacian, noise)
        field = field - (mass @ field) / mass.sum()
    return field / np.abs(field).max()


def make_dataset(domain_name, system, seed=SEED):
    rng = np.random.default_rng(seed)
    dom = domains[domain_name]
    n = dom.n_points
    p = PDE_SETTINGS
    u0_list, uT_list = [], []

    for _ in range(N_TOTAL):
        u0 = random_surface_field(rng, domain_name)

        if system in ODE_SETTINGS:
            s = ODE_SETTINGS[system]
            ode = DrivenPendulum(c=s["c"]) if system == "Pendulum" else DuffingOscillator(c=s["c"])
            u0 = s["amplitude"] * u0
            solution = PointwiseODE(ode).solve([0.0, s["T"]], state0=np.concatenate([u0, 0 * u0]),
                                               t_eval=[s["T"]], rtol=1e-8, atol=1e-8)
            uT = solution.y[:n, -1]
        elif system == "Beam":
            equation = BeamEquation(dom, EI=p["EI"], rho_A=p["rho_A"])
            equation.set_initial_condition(w0=u0, wdot0=np.zeros_like(u0))
            uT = equation.solve([0.0, p["T_beam"]], t_eval=[p["T_beam"]]).y[:n, -1]
        else:
            equation = (DiffusionEquation(dom, D=p["D"]) if system == "Diffusion"
                        else ReactionDiffusion(dom, D=p["D"], k=p["k"]))
            equation.set_initial_condition(y0=u0)
            uT = equation.solve([0.0, p["T"]], t_eval=[p["T"]]).y[:, -1]

        u0_list.append(u0)
        uT_list.append(uT)

    u0 = torch.tensor(np.array(u0_list), dtype=torch.float32).unsqueeze(-1)
    uT = torch.tensor(np.array(uT_list), dtype=torch.float32).unsqueeze(-1)
    return u0, uT


def outside_basis(U, domain_name):
    _, Phi, mass = bases[domain_name]
    U = U[..., 0].numpy().T.astype(np.float64)
    projection = Phi @ (Phi.T @ (mass[:, None] * U))
    return np.sqrt((mass[:, None] * (U - projection) ** 2).sum() / (mass[:, None] * U ** 2).sum())


datasets, data_stats = {}, {}
t0 = time.time()
for domain_name in DOMAIN_NAMES:
    for system in SYSTEM_NAMES:
        X, Y = make_dataset(domain_name, system)
        datasets[(domain_name, system)] = (X, Y)
        stats = {"change": (torch.norm(Y - X) / torch.norm(X)).item(),
                 "uT_outside_basis": outside_basis(Y, domain_name)}
        data_stats[f"{domain_name}/{system}"] = stats
        print(f"{domain_name:7s} {system:20s} ||u(T)-u(0)||/||u(0)|| {stats['change']:.3f}  "
              f"u(T) outside 96 modes {stats['uT_outside_basis']:.4f}  finite {torch.isfinite(Y).all().item()}")
print(f"data generated in {time.time() - t0:.0f}s  (IC_TYPE = {IC_TYPE})")

### Normalisation (unchanged from v2)

In [ ]:
def normalize(X, Y, n_train=N_TRAIN):
    x_mean, x_std = X[:n_train].mean(), X[:n_train].std().clamp(min=1e-8)
    y_mean, y_std = Y[:n_train].mean(), Y[:n_train].std().clamp(min=1e-8)
    return (X - x_mean) / x_std, (Y - y_mean) / y_std


def add_coordinate_channels(X, points):
    coordinates = torch.tensor(points / np.abs(points).max(), dtype=X.dtype)
    return torch.cat([X, coordinates.unsqueeze(0).expand(X.shape[0], -1, -1)], dim=-1)


for key, (X, Y) in datasets.items():
    X, Y = normalize(X, Y)
    if ADD_COORDINATE_CHANNELS:
        X = add_coordinate_channels(X, domains[key[0]].points)
    datasets[key] = (X, Y)
print("normalized using training-set statistics" + (", xyz channels appended" if ADD_COORDINATE_CHANNELS else ""))

## Model builders

`Ours` and the baselines are built exactly as in v2. The ED variants use the same
eigenbasis, diffusion times and $\phi$ MLP size as `Ours`. Only the channel
structure changes: encoder, `width` channels per block, GELU, decoder.

In [ ]:
def heat_kernel(domain_name, in_channels, out_channels):
    values, vectors, mass = bases[domain_name]
    return HeatKernel(
        eigenvalues=torch.tensor(values, dtype=torch.float32),
        eigenvectors=torch.tensor(vectors, dtype=torch.float32),
        mass=mass,
        in_channels=in_channels,
        out_channels=out_channels,
        diffusion_times=make_diffusion_times(values),
    )


def build_ours(in_channels, out_channels, domain_name):
    layers = []
    for i in range(N_OPERATOR_LAYERS):
        layer_in = in_channels if i == 0 else LATENT_CHANNELS
        layer_out = out_channels if i == N_OPERATOR_LAYERS - 1 else LATENT_CHANNELS
        layers.append(IntegralOperator(heat_kernel(domain_name, layer_in, layer_out), HIDDEN_CHANNELS))
    return IntegralOperatorModel(layers)


def build_ours_ed(in_channels, out_channels, domain_name, pointwise_skip=False):
    layers = [
        IntegralOperator(heat_kernel(domain_name, OURS_ED_WIDTH, OURS_ED_WIDTH), HIDDEN_CHANNELS)
        for _ in range(N_OPERATOR_LAYERS)
    ]
    return EncoderDecoderOperatorModel(
        layers,
        in_channels=in_channels,
        width=OURS_ED_WIDTH,
        out_channels=out_channels,
        decoder_hidden=OURS_ED_DECODER_HIDDEN,
        pointwise_skip=pointwise_skip,
    )


def build_cno(in_channels, out_channels, domain_name):
    return CNOOperator(n_points=domains[domain_name].n_points, in_channels=in_channels,
                       out_channels=out_channels, N_layers=3, N_res=2, N_res_neck=2, channel_multiplier=16)


def build_fno(in_channels, out_channels, domain_name):
    return FNOOperator(n_points=domains[domain_name].n_points, in_channels=in_channels,
                       out_channels=out_channels, modes=16, width=64)


def build_lno(in_channels, out_channels, domain_name):
    n_points = domains[domain_name].n_points
    return LNOOperator(n_points=n_points, in_channels=in_channels, out_channels=out_channels,
                       modes=16, width=4, coords=np.linspace(0.0, 1.0, n_points))


def build_sfno(in_channels, out_channels, domain_name):
    dom = domains[domain_name]
    return SphereSFNOOperator(dom.points, dom.faces, in_channels=in_channels, out_channels=out_channels,
                              nlat=SFNO_NLAT, modes=SFNO_MODES, width=SFNO_WIDTH)


def build_fno2d(in_channels, out_channels, domain_name):
    return TorusFNO2dOperator(domains[domain_name].points, n_major=TORUS_MAJOR_SECTIONS,
                              n_minor=TORUS_MINOR_SECTIONS, major_radius=TORUS_MAJOR_RADIUS,
                              in_channels=in_channels, out_channels=out_channels,
                              modes=FNO2D_MODES, width=FNO2D_WIDTH)


# name -> (builder, domains it applies to)
ALL_BUILDERS = {
    "Ours": (build_ours, ALL_DOMAINS),
    "Ours-ED": (build_ours_ed, ALL_DOMAINS),
    "Ours-ED+skip": (lambda i, o, d: build_ours_ed(i, o, d, pointwise_skip=True), ALL_DOMAINS),
    "CNO": (build_cno, ALL_DOMAINS),
    "FNO": (build_fno, ALL_DOMAINS),
    "LNO": (build_lno, ALL_DOMAINS),
    "SFNO": (build_sfno, ["Sphere"]),
    "FNO2d": (build_fno2d, ["Torus"]),
}
MODEL_BUILDERS = {name: ALL_BUILDERS[name] for name in MODELS}
MICRO_BATCH = {"Ours": OURS_MICRO_BATCH, "Ours-ED": OURS_ED_MICRO_BATCH, "Ours-ED+skip": OURS_ED_MICRO_BATCH}

## Training loop (unchanged from v2)

In [ ]:
def relative_l2(prediction, target):
    return (torch.norm(prediction - target) / torch.norm(target)).item()


@torch.no_grad()
def predict(model, X, micro_batch):
    return torch.cat([model(chunk) for chunk in X.split(micro_batch)])


def train_model(model, X, Y, micro_batch=BATCH_SIZE, n_epochs=N_EPOCHS, seed=SEED):
    torch.manual_seed(seed)
    model = model.to(device)
    X_train, Y_train = X[:N_TRAIN].to(device), Y[:N_TRAIN].to(device)
    X_test, Y_test = X[N_TRAIN:].to(device), Y[N_TRAIN:].to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, n_epochs // 4), gamma=0.5)
    history = {"train_loss": [], "test_error": []}
    n_batches = max(1, N_TRAIN // BATCH_SIZE)

    for epoch in range(n_epochs):
        model.train()
        permutation = torch.randperm(N_TRAIN, device=device)
        epoch_loss = 0.0
        for b in range(n_batches):
            index = permutation[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            optimizer.zero_grad()
            for chunk in index.split(micro_batch):
                loss = nn.functional.mse_loss(model(X_train[chunk]), Y_train[chunk]) * (len(chunk) / len(index))
                loss.backward()
                epoch_loss += loss.item()
            optimizer.step()
        scheduler.step()

        model.eval()
        history["train_loss"].append(epoch_loss / n_batches)
        history["test_error"].append(relative_l2(predict(model, X_test, micro_batch), Y_test))
    return history

## Run the benchmark

In [ ]:
results, histories, trained_models = {}, {}, {}
os.makedirs("results", exist_ok=True)
results_path = f"results/ed_benchmark_{IC_TYPE}_{run_id}.json"


def save_results():
    # written after every model, so a job that hits its time limit keeps what it finished
    with open(results_path, "w") as f:
        json.dump({
            "ic_type": IC_TYPE,
            "n_epochs": N_EPOCHS,
            "ours_ed_width": OURS_ED_WIDTH,
            "data_stats": data_stats,
            "results": {f"{d}/{s}": v for (d, s), v in results.items()},
            "histories": {f"{d}/{s}": v for (d, s), v in histories.items()},
        }, f, indent=2)


for (domain_name, system), (X, Y) in datasets.items():
    key = (domain_name, system)
    results[key], histories[key], trained_models[key] = {}, {}, {}
    print(f"\n=== {domain_name} / {system} [{IC_TYPE}] (V={X.shape[1]}, in={X.shape[-1]}) ===", flush=True)

    for model_name, (builder, applies_to) in MODEL_BUILDERS.items():
        if domain_name not in applies_to:
            continue
        torch.manual_seed(SEED)
        model = builder(X.shape[-1], Y.shape[-1], domain_name)
        n_parameters = sum(p.numel() for p in model.parameters())
        micro_batch = MICRO_BATCH.get(model_name, BATCH_SIZE)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        start = time.time()
        try:
            history = train_model(model, X, Y, micro_batch=micro_batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"  {model_name:12s} OUT OF MEMORY - lower its micro-batch", flush=True)
            history = {"train_loss": [float("nan")], "test_error": [float("nan")]}
        elapsed = time.time() - start

        results[key][model_name] = {
            "final_error": history["test_error"][-1],
            "best_error": min(history["test_error"]),
            "n_parameters": n_parameters,
            "seconds": elapsed,
        }
        histories[key][model_name] = history
        trained_models[key][model_name] = model.cpu()
        save_results()

        peak = (f"  peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB"
                if torch.cuda.is_available() else "")
        print(f"  {model_name:12s} params={n_parameters:8d}  final rel L2 {history['test_error'][-1]:.4f}  "
              f"best {min(history['test_error']):.4f}  ({elapsed:.0f}s){peak}", flush=True)

print("saved", results_path)

## Results of this job

Test relative L2 error (lower is better). The best model per row is marked `*`.
`-` means the model is not defined on that domain or was not selected.

In [ ]:
def print_table(table, model_names, value="final_error", fmt="{:>10.4f}"):
    header = f"{'Domain':<8}{'System':<20}" + "".join(f"{m:>13}" for m in model_names)
    print(header)
    print("-" * len(header))
    for key, row_results in table.items():
        row = f"{key[0]:<8}{key[1]:<20}"
        values = {m: r[value] for m, r in row_results.items()}
        finite = {m: v for m, v in values.items() if np.isfinite(v)}
        best = min(finite, key=finite.get) if finite and value == "final_error" else None
        for m in model_names:
            if m in values:
                row += "  " + fmt.format(values[m]) + ("*" if m == best else " ")
            else:
                row += f"{'-':>13}"
        print(row)


model_names = list(MODEL_BUILDERS)
print(f"IC_TYPE = {IC_TYPE}")
print_table(results, model_names)
print("\nParameter counts")
print_table(results, model_names, value="n_parameters", fmt="{:>10d}")
print("\nTraining time [s]")
print_table(results, model_names, value="seconds", fmt="{:>10.0f}")

## Loss curves

In [ ]:
colors = {"Ours": "#d62728", "Ours-ED": "#e377c2", "Ours-ED+skip": "#7f7f7f", "CNO": "#1f77b4",
          "FNO": "#2ca02c", "LNO": "#ff7f0e", "SFNO": "#9467bd", "FNO2d": "#8c564b"}

for domain_name in DOMAIN_NAMES:
    fig, axes = plt.subplots(2, len(SYSTEM_NAMES), figsize=(4.2 * len(SYSTEM_NAMES), 7.5), squeeze=False)
    for column, system in enumerate(SYSTEM_NAMES):
        for model_name, history in histories[(domain_name, system)].items():
            axes[0, column].semilogy(history["train_loss"], label=model_name, color=colors[model_name], lw=1.2)
            axes[1, column].semilogy(history["test_error"], label=model_name, color=colors[model_name], lw=1.2)
        axes[0, column].set_title(f"{domain_name} / {system} [{IC_TYPE}]")
        axes[1, column].set_xlabel("Epoch")
        for ax in axes[:, column]:
            ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel("Training MSE")
    axes[1, 0].set_ylabel("Test relative L2")
    axes[0, 0].legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(f"results/ed_loss_curves_{IC_TYPE}_{domain_name}_{run_id}.png", dpi=150)
    plt.show()

## Example predictions on the surface

In [ ]:
sample = 0
for domain_name in DOMAIN_NAMES:
    points = domains[domain_name].points
    names = [m for m in model_names if m in trained_models[(domain_name, SYSTEM_NAMES[0])]]
    columns = ["Ground truth"] + names
    fig = plt.figure(figsize=(3.0 * len(columns), 3.0 * len(SYSTEM_NAMES)))

    for row, system in enumerate(SYSTEM_NAMES):
        X, Y = datasets[(domain_name, system)]
        fields = {"Ground truth": Y[N_TRAIN + sample, :, 0].numpy()}
        for model_name in names:
            model = trained_models[(domain_name, system)][model_name].eval()
            with torch.no_grad():
                fields[model_name] = model(X[N_TRAIN + sample:N_TRAIN + sample + 1])[0, :, 0].numpy()

        vmin, vmax = fields["Ground truth"].min(), fields["Ground truth"].max()
        for column, name in enumerate(columns):
            ax = fig.add_subplot(len(SYSTEM_NAMES), len(columns), row * len(columns) + column + 1, projection="3d")
            ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=fields[name], cmap="RdBu_r",
                       vmin=vmin, vmax=vmax, s=8)
            ax.set_box_aspect(np.ptp(points, axis=0))
            ax.set_axis_off()
            error = "" if name == "Ground truth" else (
                f"\nrel L2 {np.linalg.norm(fields[name] - fields['Ground truth']) / np.linalg.norm(fields['Ground truth']):.3f}")
            ax.set_title((f"{system}: " if column == 0 else "") + name + error, fontsize=9)

    fig.suptitle(f"{domain_name} [{IC_TYPE}]")
    plt.tight_layout()
    fig.savefig(f"results/ed_predictions_{IC_TYPE}_{domain_name}_{run_id}.png", dpi=150)
    plt.show()

## Combined table across jobs

This merges every `results/ed_benchmark_<IC_TYPE>_*.json` in `results/`, so you
can split the benchmark over several jobs (by `DOMAINS` / `MODELS`) and run this
notebook once more at the end, e.g. with `BENCHMARK_SMOKE=1`, or just this cell
locally after fetching the JSON files. When a (domain, system, model) appears in
several files, the newest file wins. Smoke runs are skipped.

In [ ]:
combined = {}
for path in sorted(glob.glob(f"results/ed_benchmark_{IC_TYPE}_*.json"), key=os.path.getmtime):
    with open(path) as f:
        saved = json.load(f)
    if saved.get("n_epochs", 0) < 100:
        continue
    for key, row in saved["results"].items():
        combined.setdefault(tuple(key.split("/")), {}).update(row)

if combined:
    order = {(d, s): i for i, (d, s) in enumerate((d, s) for d in ALL_DOMAINS for s in ALL_SYSTEMS)}
    combined = dict(sorted(combined.items(), key=lambda kv: order.get(kv[0], 99)))
    present = [m for m in ALL_MODELS if any(m in row for row in combined.values())]
    print(f"IC_TYPE = {IC_TYPE}, merged from results/ed_benchmark_{IC_TYPE}_*.json")
    print_table(combined, present)
else:
    print("no full-length result files found yet")

## Reading the results

- **Ours vs Ours-ED** isolates the encoder/decoder: same basis, same diffusion
  times, same $\phi$ size. If Ours-ED closes most of the gap to SFNO / FNO2d, the
  original model's narrow 8-channel, LeakyReLU-only structure was the
  bottleneck.
- **Ours-ED vs Ours-ED+skip** isolates SFNO's pointwise path. Expect it to matter
  most on the ODEs, which are purely local maps.
- **ODEs vs PDEs.** The ODEs reward a strong local path, while diffusion and beam
  reward the spectral part. A model that wins on one group and loses on the
  other says which half of the architecture is weak.
- **Cost:** check the training-time table. The ED variants pay for `width`
  channels over all $V^2$ vertex pairs in every layer.